# Comparar execuções

Exploratório: *terá uma alteração ajudado?* Existe **uma referência por população**, partilhada por todas as *runs* de painel sobre ela, pelo que, mantendo a população fixa e mudando um único parâmetro (prompt, taxonomia, contexto de jogo, painel, temperatura), ambas as *runs* são pontuadas contra o *mesmo gold* adjudicado. A diferença de F1 é então atribuível a esse parâmetro.

**Exporte com `?run=all`** para que o zip contenha todas as *runs*; escolhemos duas.

In [1]:
%run 00-setup.ipynb   # funções partilhadas e configuração da suite

setup pronto: run 2 | zip: analysis-export.zip


## Carregar duas *runs* de uma exportação `?run=all`

In [2]:
RUN_A, RUN_B = 2, 8   # referencia, variante -- ambas sobre a mesma populacao
annA, goldA = load("annotations.csv", run=RUN_A), load("gold.csv", run=RUN_A)
annB, goldB = load("annotations.csv", run=RUN_B), load("gold.csv", run=RUN_B)
metaA = load("meta.csv", run=RUN_A).iloc[0].to_dict()
metaB = load("meta.csv", run=RUN_B).iloc[0].to_dict()
print(RUN_A, "->", RUN_B)

2 -> 8


## Funções auxiliares
O `cells` e a `panel_majority` vêm do `00-setup.ipynb`; só o `f1_by_pattern` é específico desta comparação.

In [3]:
from sklearn.metrics import precision_recall_fscore_support
def f1_by_pattern(ann, gold, pass_="open"):
    d = cells(ann, gold, pass_)
    rows = []
    for (model, code), g in d.groupby(["model","code"]):
        _, _, f, _ = precision_recall_fscore_support(g["y_true"], g["y_pred"], average="binary", pos_label=1, zero_division=0)
        rows.append({"model":model,"code":code,"f1":round(f,3)})
    return pd.DataFrame(rows).set_index(["model","code"])["f1"]

## A comparação factorial: diferença de F1 por par (modelo, padrão), mesma referência
Uma diferença positiva = a *run* B foi melhor. `changed_knobs` reporta o que difere, ou seja, aquilo a que se atribui a diferença. O código recusa *runs* sobre populações diferentes.

In [4]:
KNOBS = ["taxonomyVersion","prompt","gameContext","temperature","panel"]
def changed_knobs(a, b): return {k:(a.get(k), b.get(k)) for k in KNOBS if a.get(k) != b.get(k)}
if metaA["populationId"] != metaB["populationId"]:
    raise ValueError("different populations -- the gold is not shared, this is not an ablation")

print("alterado:", changed_knobs(metaA, metaB))
fa, fb = f1_by_pattern(annA, goldA), f1_by_pattern(annB, goldB)
df = pd.concat([fa.rename("f1_a"), fb.rename("f1_b")], axis=1).fillna(0.0)
df["delta"] = (df["f1_b"] - df["f1_a"]).round(3)
print("\n-- delta macro por modelo (ajudou em media?) --")
print(df.groupby(level="model")[["f1_a","f1_b","delta"]].mean().round(3).to_string())
print("\n-- maiores oscilacoes (modelo, padrao) --")
print(pd.concat([df.sort_values("delta").head(5), df.sort_values("delta").tail(5)]).to_string())

alterado: {'prompt': ('enhanced prompt v2', 'text-context v2'), 'gameContext': (False, True)}



-- delta macro por modelo (ajudou em media?) --
                                    f1_a   f1_b  delta
model                                                 
deepseek/deepseek-v4-flash         0.204  0.179 -0.026
meta-llama/llama-3.3-70b-instruct  0.272  0.183 -0.089
openai/gpt-oss-120b                0.179  0.136 -0.043
panel-majority                     0.194  0.114 -0.081
qwen/qwen3-next-80b-a3b-instruct   0.446  0.400 -0.046

-- maiores oscilacoes (modelo, padrao) --
                                         f1_a   f1_b  delta
model                             code                     
qwen/qwen3-next-80b-a3b-instruct  DR-1  0.500  0.000 -0.500
panel-majority                    DR-1  0.333  0.000 -0.333
meta-llama/llama-3.3-70b-instruct SE-3  0.417  0.091 -0.326
                                  PM-1  0.769  0.485 -0.284
                                  TM-1  0.652  0.381 -0.271
qwen/qwen3-next-80b-a3b-instruct  PE-2  0.536  0.581  0.045
                                  PM-1  0.6

## A face sem *gold*: divergência
Com que frequência as maiorias dos dois painéis simplesmente discordam, ao longo das revisões que ambos correram. Não necessita da referência: não diz qual a *run* melhor, apenas o quão afastadas estão.

In [5]:
def majorities(ann): return panel_majority(ann).set_index(["individualId","code"])["present"]
j = pd.concat([majorities(annA).rename("a"), majorities(annB).rename("b")], axis=1).dropna()
print(f"divergencia: {int((j['a'] != j['b']).sum())} / {len(j)} celulas em que as maiorias dos dois paineis discordam")

divergencia: 68 / 17100 celulas em que as maiorias dos dois paineis discordam


> O *gold*, aqui, é o *consenso adjudicado*, adjudicado contra um painel. Reutilizá-lo para pontuar um painel diferente é legítimo na passagem aberta, desde que se leia a diferença como **«mais próximo do consenso do autor»**, e não «mais próximo de uma verdade independente». Se os dois painéis não partilham modelos (por exemplo, se trocou o painel inteiro), só a `panel-majority` alinha modelo a modelo: leia a macro nessa linha.

## Prevalência entre *runs*: com uma regra *justa face ao tamanho do painel*
Comparar com que frequência cada padrão é sinalizado entre *runs* só é justo se a **regra de agregação for a mesma**. O senão: uma *maioria estrita* representa um patamar diferente consoante o tamanho do painel (dois em três numa *run* de 3 modelos, três em quatro numa *run* de 4), pelo que um painel mais pequeno parece, sem mérito próprio, mais propenso a sinalizar. O `panel_verdict` torna a regra **configurável**, para que se possa mantê-la fixa entre painéis de tamanhos diferentes.

In [6]:
def panel_verdict(ann, rule="strict"):
    """Um veredicto por (revisao, padrao). Configuravel para que paineis de tamanhos DIFERENTES sejam comparaveis:
       "strict" -> votes > count/2   (maioria estrita; DEPENDENTE do tamanho: 2-de-3 vs 3-de-4)
       "half"   -> votes >= count/2  (pelo menos metade)
       int k    -> votes >= k        (absoluto: MESMO patamar de evidencia para qualquer tamanho)  <- justo entre tamanhos
       float f  -> votes >= f*count  (mesma fraccao para qualquer tamanho)
    """
    g = ann.groupby(["individualId", "code"])["present"]; s, n = g.sum(), g.count()
    if rule == "strict": return s * 2 > n
    if rule == "half":   return s * 2 >= n
    if isinstance(rule, int):   return s >= rule
    if isinstance(rule, float): return s >= rule * n
    raise ValueError(f"unknown rule {rule!r}")

RUNS = [1, 2, 8, 9]   # runs a comparar (tem de partilhar populacao); 1/2/8 sao de 4 modelos, 9 e de 3 (claude descontinuado)
RULE = 2              # absoluto "pelo menos k concordam" -> mesmo patamar quer o painel tenha 3 ou 4 membros
prev = {}
for r in RUNS:
    ann_r = load("annotations.csv", run=r)
    prev[f"run{r}"] = panel_verdict(ann_r, RULE).groupby("code").sum()
tab = pd.DataFrame(prev).fillna(0).astype(int)
tab = tab.loc[tab.sum(axis=1).sort_values(ascending=False).index]
print(f"revisoes sinalizadas por padrao, regra={RULE!r} (mesmo patamar entre tamanhos de painel) -- totais:",
      {c: int(tab[c].sum()) for c in tab.columns})
tab.head(10)

revisoes sinalizadas por padrao, regra=2 (mesmo patamar entre tamanhos de painel) -- totais: {'run1': 411, 'run2': 232, 'run8': 188, 'run9': 242}


,run1,run2,run8,run9
code,,,,
TM-1,95,65,46,74
PM-1,84,66,56,63
PE-1,24,21,19,12
PE-2,42,10,6,16
DR-2,23,14,8,9
PM-3,16,15,10,10
SE-3,22,9,9,10
PE-3,27,5,3,13
PM-4,17,5,8,12


> **Porquê um `k` absoluto e não `strict`.** Sob maioria estrita, a *run* 9 (um painel de 3 modelos, logo dois em três) sinaliza muito mais do que a *run* 8 (um painel de 4 modelos, três em quatro), em parte *apenas por o seu patamar ser mais baixo*. Mude `RULE` para `2` (o mesmo «pelo menos dois modelos concordam» para todos) e a comparação torna-se honesta: o painel comercial continua a sinalizar mais do que o aberto, mas a parte inflacionada da diferença desaparece. **Prevalência não é validade**: mais sinalizações podem significar mais detecções verdadeiras *ou* mais falsos alarmes; só a calibração contra a referência (as diferenças de F1 acima) o diz. Leia-se esta tabela quanto à *cobertura*, e a calibração quanto à *correcção*. Para uma leitura independente do tamanho, compare antes as taxas de positivos por modelo (cada modelo julgado isoladamente).